# Reliable DFU Framework V2 — storage-bounded grouped CV
Fresh run ID `RELIABLE_DFU_CV_V2`. Incomplete trials keep exact optimizer-resume checkpoints. Completed trials retain verified FP16 model weights, predictions, calibration parameters, histories and metrics, then remove bulky completed optimizer states.

In [ ]:
import os, sys, shutil, subprocess, json, time, uuid
from pathlib import Path
from google.colab import drive

MOUNT=Path('/content/drive'); MY=MOUNT/'MyDrive'
if not MY.is_dir(): drive.mount(str(MOUNT), force_remount=False)
if not MY.is_dir(): raise RuntimeError('Google Drive unavailable; training not started.')
probe=MY/'DFU-ImageGuard'/'_mount_verification'/f'v2_{uuid.uuid4().hex}.txt'
probe.parent.mkdir(parents=True,exist_ok=True); value=str(time.time_ns()); probe.write_text(value)
if probe.read_text()!=value: raise RuntimeError('Drive write/read verification failed.')
probe.unlink(); print('Drive write/read verification: PASS')

subprocess.run([sys.executable,'-m','pip','install','-q','timm>=1.0.9','kagglehub>=0.3','ImageHash>=4.3','scikit-learn>=1.5','scipy>=1.13','matplotlib>=3.9','pandas>=2.2','Pillow>=10.4','tabulate>=0.9'],check=True)
REPO='https://github.com/AzizulHakim00/DFU-ImageGuard.git'
PINNED_COMMIT='349143b4d8b16f885adce3559542f6c202a2bca1'
WORK=Path('/content/DFU-ImageGuard-reliable-v2')
if WORK.exists(): shutil.rmtree(WORK)
subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO,str(WORK)],check=True)
subprocess.run(['git','-C',str(WORK),'checkout',PINNED_COMMIT],check=True)
os.chdir(WORK); sys.path.insert(0,str(WORK))
for module_name in list(sys.modules):
    if module_name=='src' or module_name.startswith('src.'):
        del sys.modules[module_name]
commit=subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
if commit!=PINNED_COMMIT: raise RuntimeError(f'Commit mismatch: {commit} != {PINNED_COMMIT}')
print('Loaded reliable framework V2 commit:',commit)
from src.reliable_runner_v2 import ReliableSettingsV2, run_reliable_framework_v2
settings=ReliableSettingsV2(run_id='RELIABLE_DFU_CV_V2',seeds=(2026,2027,2028),folds=(0,1,2,3,4),models=('convnextv2_tiny','mobilenetv3_large','densenet121'),max_epochs=30,patience=7,batch_size=16,num_workers=2,target_sensitivity=.95,source_commit=commit)
result=run_reliable_framework_v2(settings)
print(json.dumps(result,indent=2,default=str))
